# 3. Versioning data

This is the Binder copy of [the full page](dataops.ipynb).

## Learning objectives

- Version a data file with DVC: the data goes to a bucket, and a small pointer file goes in Git.
- Make a new data version, rerun the declared steps, and compare the results with `dvc metrics diff`.
- Go back to an old data version with its numbers and its figure, with Git and `dvc checkout`.

## Builds on Exercise 2

- The project starts from [Exercise 2](../02-workflows/workflows.ipynb)'s finished workflow, with the `plot` step and the README's figure.
- DVC stages are the same declared steps, plus versioned data and versioned outputs.

## The problem

- The workshop's exercises use one project, [a study of Palmer penguins](../../project.md), and add a capability in each exercise.
- The project reports how much heavier male penguins are than females: male mean body mass minus female mean body mass, per species, in grams.
- `data/penguins.csv` holds the 2007 field season, and it's in Git. The 2008 and 2009 seasons are coming.
- Git keeps a full copy of every version of every file in every clone. That's fine for a 15 KB table, and not for a 50 GB dataset.
- Each result, the numbers and the figure, needs to point to the exact data it came from, so you can get them back later.

## What you'll do

1. Step 1: move the 2007 data from Git to DVC, push it to the bucket, and commit the pointer.
2. Step 2: take the 2008 and 2009 seasons as a new data version, rerun the pipeline, commit and push.
3. Step 3: go back to the 2007 version, its numbers and its figure, then return to the latest.
4. Run one check.

- The whole exercise takes about 15 minutes.
- Only reading? Each step shows what its commands print.

## The rule

- The data is a pet. Each version has a pointer in Git and a copy in the bucket, and nobody edits a version in place.
- Everything built is cattle: `results/`, `metrics.json` and `figures/`. None of it is in Git. `dvc repro` rebuilds it, and DVC keeps each version in its cache, `.dvc/cache`, and the bucket, so `dvc checkout` or `dvc pull` brings an old one back without a rebuild.

## In Binder

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from a real run of these commands.

- The terminal is on the left, already in the `project` folder. A read-only copy of this page is on the right.
- Copy and paste a command: Ctrl+C, then Ctrl+V in the terminal. On a Mac, Cmd+C and Cmd+V.
- A session is temporary and has 2 GB of memory. [Download anything you want to keep](../../setup.md#download-your-work).


**Editing files**

- In Binder, double-click a file in the file panel on the left to open it in JupyterLab's editor. Save with Ctrl+S, or Cmd+S on a Mac.
- Or edit it in the terminal with `nano <file>`, `vi <file>` or `vim <file>`. Binder has all three.
- nano is the simplest. Ctrl+O, then Enter, saves. Ctrl+X quits.

**With uv instead of conda**

- The terminal starts in the exercise's conda environment, and `project` is already built. To use uv instead, build `.venv` from the exercise's `uv.lock` in the folder above `project`, and activate it:

**Run:**

```bash
cd ~
uv sync --locked
source .venv/bin/activate
cd project
```

- You don't run `scripts/setup_project.py`. Binder ran it when it built the session, and a second run leaves `project` as it is.
- `uv sync` downloads Python 3.12.12 and the locked packages. It took 7 to 11 seconds on mybinder.org.
- A new terminal starts in conda's environment again. Run `source ~/.venv/bin/activate` in it.

- DVC may print "Waiting to acquire lock. If DVC froze, see hardlink_lock" several times. It's harmless, and the command finishes.

## The project

- `project` is a Git repository with two commits. The exercise's setup script, `scripts/setup_project.py`, made both.

- On Binder, the script ran before your session opened.

- The first is the end of Exercise 2: the 2007 data, `clean.py`, `validate.py`, `report.py`, `plot.py`, `params.yaml`, the README and the `Snakefile` with the `plot` rule, all in Git. Nothing built is in Git.
- The second declares the same four steps as DVC stages, in `dvc.yaml`. The data is still in Git. The outputs, including `metrics.json` and `figures/body_mass.svg`, are in DVC's cache, and `dvc.lock` records their hashes.
- Your commit IDs from the second commit on can differ from the ones on this page. `dvc.lock` holds the figure's hash, and the figure's bytes depend on the font library in your environment.
- `../osn-bucket` stands in for an object storage bucket such as OSN. DVC uses a real bucket the same way, and your keys stay in `--local` config, out of Git.

**The commands the setup script ran for the second commit.** You don't run them here. Run them to set up DVC on your own project.

- `dvc stage add` declares a step: what it reads, what it writes, and which section of `params.yaml` it uses. `-o` and `-m` outputs go to DVC's cache, not Git. `-m` marks `metrics.json` as metrics, for `dvc metrics`.
- `dvc repro` runs whatever is out of date, and records the hash of each input and output in `dvc.lock`.

```text
dvc init
dvc remote add -d storage ../osn-bucket
dvc stage add -n clean -d data/penguins.csv -d clean.py -o results/clean.csv python clean.py
dvc stage add -n validate -d results/clean.csv -d validate.py -o results/validated.csv python validate.py
dvc stage add -n report -d results/validated.csv -d report.py -p report -m metrics.json python report.py
dvc stage add -n plot -d results/validated.csv -d plot.py -o figures/body_mass.svg python plot.py
dvc repro
git add .
git commit -m "Declare the same steps as DVC stages"
```

Look at the project:

**Run:**

```bash
git log --format='%h  %s'
cat dvc.yaml
dvc metrics show
```

**Recorded output:**

```text
$ git log --format='%h  %s'
9cd0c43  Declare the same steps as DVC stages
faaa3a5  Build the numbers and the figure with a Snakefile

$ cat dvc.yaml
stages:
  clean:
    cmd: python clean.py
    deps:
    - clean.py
    - data/penguins.csv
    outs:
    - results/clean.csv
  validate:
    cmd: python validate.py
    deps:
    - results/clean.csv
    - validate.py
    outs:
    - results/validated.csv
  report:
    cmd: python report.py
    deps:
    - report.py
    - results/validated.csv
    params:
    - report
    metrics:
    - metrics.json
  plot:
    cmd: python plot.py
    deps:
    - plot.py
    - results/validated.csv
    outs:
    - figures/body_mass.svg

$ dvc metrics show
Path          dimorphism_g_adelie    dimorphism_g_chinstrap    dimorphism_g_gentoo    n_rows
metrics.json  648.9                  250.0                     934.2                  103
```

---

## Step 1: put the data under DVC

**Problem:** `data/penguins.csv` is in Git, so every version of it goes into every clone.

**Facts**

- `git rm --cached` stops Git tracking a file, and leaves the file in place.
- `dvc add` copies the file into DVC's cache, and writes a pointer file, `data/penguins.csv.dvc`, with the file's hash. It also adds the data file to `data/.gitignore`.
- Git tracks the pointer. `dvc push` copies the data from the cache to the bucket.
- A Git tag names a commit, so you can come back to it in step 3.
- DVC prints `To track the changes with git, run: ...`. The commands below do that.

**Steps**

1. Move the data from Git to DVC, commit the pointer, tag the commit and push the data.

**Run:**

```bash
git rm --cached data/penguins.csv
dvc add data/penguins.csv
git add data/penguins.csv.dvc data/.gitignore
git commit -m "Track the 2007 season with DVC"
git tag season-2007
dvc push
```

**Recorded output:**

```text
$ git rm --cached data/penguins.csv
rm 'data/penguins.csv'

$ dvc add data/penguins.csv

To track the changes with git, run:

	git add data/penguins.csv.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true

$ git add data/penguins.csv.dvc data/.gitignore

$ git commit -m "Track the 2007 season with DVC"
[main ebb4399] Track the 2007 season with DVC
 3 files changed, 6 insertions(+), 111 deletions(-)
 create mode 100644 data/.gitignore
 delete mode 100644 data/penguins.csv
 create mode 100644 data/penguins.csv.dvc

$ git tag season-2007

$ dvc push
4 files pushed
```

- DVC's `git add` suggestion can list the two files in the other order on your machine.
- `dvc push` sent four files: the data, `metrics.json`, the figure and the cleaned table. `results/clean.csv` and `results/validated.csv` have the same contents here, so DVC stores one copy.

---

2. Look at the pointer, what Git tracks in `data`, and the bucket.

**Run:**

```bash
cat data/penguins.csv.dvc
git ls-files data
find ../osn-bucket -type f
```

**Recorded output:**

```text
$ cat data/penguins.csv.dvc
outs:
- md5: 0f65687151ae03322b262c13b51d2fdc
  size: 4934
  hash: md5
  path: penguins.csv

$ git ls-files data
data/.gitignore
data/penguins.csv.dvc

$ find ../osn-bucket -type f
../osn-bucket/files/md5/57/cb81982bab276d65e5b1a25767426e
../osn-bucket/files/md5/0f/65687151ae03322b262c13b51d2fdc
../osn-bucket/files/md5/6f/d7bd68f89203e196d92e70579c5f4c
../osn-bucket/files/md5/d1/018e536c75c05a335b98e489e7d806
```

- The pointer holds the file's MD5 hash and size. The bucket stores the file under that hash.
- `find` can list the bucket's files in another order on your machine.
- The two earlier commits still hold the 2007 file in Git. From this commit on, new versions go to the bucket.

---

## Step 2: a new data version

**Problem:** the 2008 and 2009 seasons arrive, and the result has to come from them.

**Facts**

- `new_season.py` replaces `data/penguins.csv` with all three seasons, 344 penguins.
- `dvc add` stores the new version in the cache, and updates the hash in the pointer. The old version stays in the cache and the bucket.
- `dvc repro` sees that the data for `clean` changed, and reruns `clean` and every stage after it, `plot` included. The figure changes with the numbers, and `dvc.lock` records the new hashes of both.
- `dvc metrics diff` compares `metrics.json` with your last commit's, which DVC reads from its cache.

**Steps**

1. Take the new seasons, update the pointer, rerun the pipeline and compare.

**Run:**

```bash
python ../scripts/new_season.py
dvc add data/penguins.csv
dvc repro
dvc metrics diff
```

**Recorded output:**

```text
$ python ../scripts/new_season.py
Delivered 344 rows to project/data/penguins.csv, covering 2007 to 2009.

$ dvc add data/penguins.csv

To track the changes with git, run:

	git add data/penguins.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

$ dvc repro
'data/penguins.csv.dvc' didn't change, skipping
Running stage 'clean':
> python clean.py
clean: kept 333 of 344 penguins
Updating lock file 'dvc.lock'

Running stage 'validate':
> python validate.py
validate: 333 rows pass
Updating lock file 'dvc.lock'

Running stage 'report':
> python report.py
{
  "dimorphism_g_adelie": 674.7,
  "dimorphism_g_chinstrap": 411.8,
  "dimorphism_g_gentoo": 805.1,
  "n_rows": 333
}
Updating lock file 'dvc.lock'

Running stage 'plot':
> python plot.py
plot: wrote figures/body_mass.svg from 333 penguins
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.

$ dvc metrics diff
Path          Metric                  HEAD    workspace    Change
metrics.json  dimorphism_g_adelie     648.9   674.7        25.8
metrics.json  dimorphism_g_chinstrap  250.0   411.8        161.8
metrics.json  dimorphism_g_gentoo     934.2   805.1        -129.1
metrics.json  n_rows                  103     333          230
```

- With all three seasons, Adelie males are 674.7 g heavier, Chinstrap males 411.8 g and Gentoo males 805.1 g, from 333 penguins.

- In Binder, right-click `README.md` in the file panel, choose Open With, then Markdown Preview, to see the figure from all three seasons.

---

2. Commit the pointer and the lock together, and push the new data and outputs. `git status --short` shows only those two files: `metrics.json` and the figure are DVC outputs, not in Git.

**Run:**

```bash
git status --short
git add data/penguins.csv.dvc dvc.lock
git commit -m "Add the 2008 and 2009 seasons"
dvc push
```

**Recorded output:**

```text
$ git status --short
 M data/penguins.csv.dvc
 M dvc.lock

$ git add data/penguins.csv.dvc dvc.lock

$ git commit -m "Add the 2008 and 2009 seasons"
[main 9fa6481] Add the 2008 and 2009 seasons
 2 files changed, 17 insertions(+), 17 deletions(-)

$ dvc push
4 files pushed
```

---

## Step 3: go back to the 2007 version

**Problem:** a reviewer asks for the 2007 numbers, from the 2007 data.

**Facts**

- The tag `season-2007` names the commit from step 1. Its pointer file and `dvc.lock` hold the hashes of the 2007 data, its numbers and its figure.
- `git checkout season-2007 -- <files>` puts those files back as they were in that commit.
- `dvc checkout` makes `data/penguins.csv`, `results/`, `metrics.json` and the figure match the pointer and `dvc.lock`, from the cache. Nothing reruns. The 2007 figure comes back with the 2007 numbers.
- In a fresh clone, `dvc pull` gets the same files from the bucket.

**Steps**

1. Go back to the 2007 version. The headline is the 2007 one again.

**Run:**

```bash
git checkout season-2007 -- data/penguins.csv.dvc dvc.lock
dvc checkout
dvc metrics show
```

**Recorded output:**

```text
$ git checkout season-2007 -- data/penguins.csv.dvc dvc.lock

$ dvc checkout
M       data/penguins.csv
M       figures/body_mass.svg
M       metrics.json
M       results/clean.csv
M       results/validated.csv

$ dvc metrics show
Path          dimorphism_g_adelie    dimorphism_g_chinstrap    dimorphism_g_gentoo    n_rows
metrics.json  648.9                  250.0                     934.2                  103
```

- The README's preview shows the 2007 figure again.

---

2. Return to the latest version. `dvc metrics diff` and `git status --short` print nothing when your files match your last commit.

**Run:**

```bash
git checkout main -- data/penguins.csv.dvc dvc.lock
dvc checkout
dvc metrics diff
git status --short
```

**Recorded output:**

```text
$ git checkout main -- data/penguins.csv.dvc dvc.lock

$ dvc checkout
M       data/penguins.csv
M       figures/body_mass.svg
M       metrics.json
M       results/clean.csv
M       results/validated.csv

$ dvc metrics diff

$ git status --short
```

## Check your work

- Run this when all three steps are done. Each line says what to do next.
- It clones your last commit into a throwaway folder, with an empty DVC cache.
- It pulls each data version from your bucket with its `metrics.json` and its figure, and checks both against the data the commit points to. It takes under a minute.

**Run:**

```bash
python ../check.py
```

On the starter project it fails like this:

**Recorded output:**

```text
$ python ../check.py
FAIL  Your last commit has no data/penguins.csv.dvc, so DVC doesn't track the data yet. Do step 1.
FAIL  There's no DVC pointer in your last commit, so a fresh clone has no data to pull. Do step 1.
FAIL  No commit holds a DVC pointer to the 2007 season. Do step 1, and commit it before step 2.
FAIL  Your last commit doesn't hold all three seasons, 2007 to 2009, from the bucket. Do step 2: commit, then dvc push.

0 of 4 parts pass.
```

## Summary

- DVC keeps the data out of Git, and a pointer to it in Git. The bucket holds every version.
- A commit ties the pointer and `dvc.lock` together. `dvc.lock` holds the hashes of `metrics.json` and the figure, so each result points to its data, and none of it is in Git.
- `dvc repro` reruns the declared steps when the data changes, and `dvc metrics diff` shows what moved.
- Git and `dvc checkout` bring back an old version with its numbers and its figure. `dvc pull` does the same in a fresh clone.

## On your own project

- `dvc add` the dataset your last result came from, and `dvc push` it. A local folder works as the remote until you have a bucket.
- Tag the commit that made a published result.
- Clone your project into an empty folder, run `dvc pull`, and check that the result comes back.

## About the data

- Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.
- `data/penguins.csv` starts as the 2007 rows, 110 penguins. `scripts/new_season.py` replaces it with the full table, 344 penguins from 2007 to 2009. Both are unchanged from the package.